# Data Preprocessing

## DeepFilterNet




It is a deep learning model that is used for noise supression.
View for more details: [GitHub Repo](https://github.com/Rikorose/DeepFilterNet)

I installed it locally and used it to filter out the background noise from the audio files. The model is trained on a large dataset of clean and noisy audio samples, allowing it to effectively separate the desired signal from the noise.

## LUFS normalization & preprocessing

According to different research papers

Links:
- [Efficient English Text-to-Speech Voice Cloning Using Limited Speaker Data](https://www.researchgate.net/publication/399217080_Efficient_English_Text-to-Speech_Voice_Cloning_Using_Limited_Speaker_Data)
- Sample Rate
- Remove silence

In [ ]:
from pydub import AudioSegment
from pydub.effects import compress_dynamic_range
import pydub.effects as effects
import numpy as np
import pyloudnorm as pyln
import soundfile as sf
import librosa
import os

In [ ]:
audio_files = glob(f'/content/drive/MyDrive/Privat/Mama/DeepFilterNet/*.wav')

for audio_filename in audio_files:
    audio = AudioSegment.from_file(audio_filename)

    audio = audio.set_channels(1).set_frame_rate(24000)
    audio = audio.high_pass_filter(80) # 80-100

    samples = np.array(audio.get_array_of_samples()).astype(np.float32)
    samples /= np.iinfo(audio.array_type).max

    samples = samples - np.mean(samples)

    meter = pyln.Meter(24000)
    loudness = meter.integrated_loudness(samples)
    target_lufs = -20
    normalized_samples = pyln.normalize.loudness(samples, loudness, target_lufs)


    peak = np.max(np.abs(normalized_samples))
    if peak > 0.99:
        normalized_samples = normalized_samples / peak * 0.99
        print(f"Peak limiting applied: {peak:.4f} → 0.99")

    output_name = os.path.splitext(os.path.basename(audio_filename))[0]
    output_path = f"/content/drive/MyDrive/Privat/Mama/DeepFilterNet/LUFUS_DFN/{output_name}.wav"
    sf.write(output_path, normalized_samples, 24000, subtype='FLOAT')


    print(f"{output_name}: {loudness:.1f} → {target_lufs:.1f} LUFS")